# CPBM — Layer 1: Pulse Signature Exploration

This notebook walks through the construction and analysis of individual pulse signatures $\varphi_i \in \mathbb{R}^7$, their clustering into behavioural archetypes, and the diagnostic plots that validate the signature space.

**Repositories:** [OSF](https://osf.io/9ukxe/) | [GitHub](https://github.com/ahmed-elsayed-99/CPBM-algorithm)  
**ORCID:** [0009-0000-5475-3970](https://orcid.org/0009-0000-5475-3970)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

from cpbm.data.synthetic import SyntheticCommunity
from cpbm.core.signature import SignatureExtractor

plt.rcParams.update({'figure.dpi': 110, 'font.size': 10})
sns.set_style('whitegrid')

## 1. Generate Synthetic Community

In [ ]:
community = SyntheticCommunity(n=500, seed=42).generate()
df        = community['dataframe']
Phi       = community['phi_matrix']
strata    = community['strata']
labels    = community['labels']
transactions = community['transactions']

print(f'Community size : {len(df)}')
print(f'Phi shape      : {Phi.shape}')
print(f'Purchase rate  : {labels.mean():.3f}')
df.groupby('stratum')[['f_score','s_score','h_entropy','label']].mean().round(3)

## 2. Extract Signatures via SignatureExtractor

In [ ]:
extractor = SignatureExtractor(window_days=90, k_range=(2, 8), random_state=42)
Phi_ext   = extractor.fit_transform(transactions)

print(f'Optimal K* = {extractor.k_star_}')
print(f'Silhouette scores: {extractor.silhouette_scores_}')
extractor.get_cluster_profiles()

## 3. Silhouette Score vs K

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
ks   = list(extractor.silhouette_scores_.keys())
sils = list(extractor.silhouette_scores_.values())
ax.plot(ks, sils, marker='o', color='#1abc9c', linewidth=2)
ax.axvline(extractor.k_star_, color='#e74c3c', linestyle='--',
           label=f'K* = {extractor.k_star_}')
ax.set_xlabel('Number of clusters K')
ax.set_ylabel('Silhouette coefficient')
ax.set_title('Optimal cluster selection — Silhouette analysis')
ax.legend()
plt.tight_layout()
plt.show()

## 4. PCA Projection — Clusters vs Strata

In [ ]:
pca  = PCA(n_components=2, random_state=42)
X_2d = pca.fit_transform(StandardScaler().fit_transform(Phi_ext))
var  = pca.explained_variance_ratio_

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

sc0 = axes[0].scatter(X_2d[:, 0], X_2d[:, 1],
                      c=extractor.cluster_labels_, cmap='tab10',
                      alpha=0.65, s=22)
plt.colorbar(sc0, ax=axes[0], label='Cluster')
axes[0].set_title('Behavioural Archetypes (K-Means)')
axes[0].set_xlabel(f'PC1 ({var[0]*100:.1f}%)')
axes[0].set_ylabel(f'PC2 ({var[1]*100:.1f}%)')

sc1 = axes[1].scatter(X_2d[:, 0], X_2d[:, 1],
                      c=strata, cmap='RdYlGn',
                      alpha=0.65, s=22)
plt.colorbar(sc1, ax=axes[1], label='Stratum')
axes[1].set_title('Socioeconomic Strata')
axes[1].set_xlabel(f'PC1 ({var[0]*100:.1f}%)')
axes[1].set_ylabel(f'PC2 ({var[1]*100:.1f}%)')

plt.suptitle('Pulse Signature Space — Clusters ≠ Strata', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()
print('Non-isomorphic mapping confirms signatures capture information beyond income stratum.')

## 5. Signature Distributions per Stratum

In [ ]:
sig_df = extractor.get_signature_dataframe()
sig_df['stratum'] = strata

features = ['f_score', 's_score', 'sigma_timing', 'e_low', 'e_high', 'c_max', 'h_entropy']
fig, axes = plt.subplots(2, 4, figsize=(16, 7))
axes = axes.flatten()
colors = {1: '#e74c3c', 2: '#2ecc71', 3: '#3498db'}

for idx, feat in enumerate(features):
    ax = axes[idx]
    for s in [1, 2, 3]:
        vals = sig_df[sig_df['stratum'] == s][feat]
        ax.hist(vals, bins=20, alpha=0.55, color=colors[s],
                label=f'Stratum {s}', density=True)
    ax.set_title(feat, fontweight='bold')
    ax.set_xlabel('value')
    ax.set_ylabel('density')
    if idx == 0:
        ax.legend(fontsize=8)

axes[-1].axis('off')
plt.suptitle('Pulse Signature Distributions by Stratum', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## 6. Timing Entropy vs Purchase Label

In [ ]:
sig_df['label'] = labels

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for lbl, color, name in [(0, '#e74c3c', 'No purchase'), (1, '#1abc9c', 'Purchased')]:
    vals = sig_df[sig_df['label'] == lbl]['h_entropy']
    axes[0].hist(vals, bins=25, alpha=0.6, color=color,
                 label=name, density=True)
axes[0].set_title('Timing Entropy H by Purchase Outcome')
axes[0].set_xlabel('H (bits)')
axes[0].legend()

axes[1].scatter(sig_df['f_score'], sig_df['h_entropy'],
               c=labels, cmap='RdYlGn', alpha=0.5, s=18)
axes[1].set_xlabel('Purchase frequency f_i')
axes[1].set_ylabel('Timing entropy H_i')
axes[1].set_title('Frequency vs Entropy (colour = label)')

plt.tight_layout()
plt.show()

## 7. Correlation Heatmap of Signature Components

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))
corr = sig_df[features + ['label']].corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f',
            cmap='RdYlGn', center=0, ax=ax,
            linewidths=0.5, linecolor='white')
ax.set_title('Pulse Signature — Correlation Matrix', fontweight='bold')
plt.tight_layout()
plt.show()